In [13]:
import json
from google.cloud import storage
import os

GCS_BUCKET = os.getenv("GCS_BUCKET")

def read_rawg_games_gcs(year: int) -> list[dict]:
    client = storage.Client()
    bucket = client.bucket(GCS_BUCKET)
    prefix = f"raw/year={year}/"

    blobs = bucket.list_blobs(prefix=prefix)

    all_games = []

    for blob in blobs:
        if not blob.name.endswith(".json"):
            continue

        content = blob.download_as_text()
        payload = json.loads(content)

        results = payload.get("results", [])
        if results:
            all_games.extend(results)

    return all_games

In [14]:
all_games = read_rawg_games_gcs(
        year=2022
    )

In [15]:
import pandas as pd

games_data = []

for game in all_games:
        game_record = {
            "game_id": game.get("id"),
            "name": game.get("name"),
            "released": game.get("released"),
            "rating": game.get("rating"),
            "ratings_count": game.get("ratings_count"),
        }
        games_data.append(game_record)

games_df = pd.DataFrame(games_data)

In [16]:
games_df.head()

,game_id,name,released,rating,ratings_count
0,326243,Elden Ring,2022-02-25,4.38,1456
1,452638,Stray,2022-07-19,4.15,1251
2,685577,Vampire Survivors,2022-10-20,4.19,791
3,704634,Uncharted: Legacy of Thieves Collection,2022-01-28,4.46,516
4,372576,Lost Ark,2022-02-11,3.18,285


In [ ]:
games_df.to_parquet(f"gs://{GCS_BUCKET}/processed/year=2020/games.parquet", index=False)